# Update Media Locations CreatedAt Column
This notebook reads all entries from the `MediaLocation` table, retrieves the creation time of each file on disk, and updates the `CreatedAt` column accordingly.

### Import dependencies and namespaces

In [ ]:
#r "nuget: Microsoft.EntityFrameworkCore.Sqlite"
#r "..\Japlayer.Data\bin\Debug\net8.0\Japlayer.Data.dll"
#r "..\Japlayer.WinUI\bin\Debug\net8.0-windows10.0.19041.0\Japlayer.dll"

using Japlayer.Data.Context;
using Japlayer.Data.Entities;
using Japlayer.Models;
using Microsoft.EntityFrameworkCore;
using System;
using System.IO;
using System.Text.Json;
using System.Threading.Tasks;
using System.Linq;

### Initialize Database Context

In [ ]:
var settingsFilePath = File.Exists(@"..\japlayer.settings.json.local")
    ? @"..\japlayer.settings.json.local"
    : @"..\japlayer.settings.json";

string jsonContent = File.ReadAllText(settingsFilePath);
var options = new JsonSerializerOptions { PropertyNameCaseInsensitive = true };
AppSettings settings = JsonSerializer.Deserialize<AppSettings>(jsonContent, options);

var optionsBuilder = new DbContextOptionsBuilder<DatabaseContext>();
optionsBuilder.UseSqlite($"Data Source={settings.SqliteDatabasePath}");

var context = new DatabaseContext(optionsBuilder.Options);
Console.WriteLine($"✓ Initialized context for: {settings.SqliteDatabasePath}");

### Ensure CreatedAt column exists and has no NULL values
If the database already exists, it won't automatically have the new `createdAt` column. We run a migration check and SQL command to add it if missing. We also update any existing `NULL` values to a default timestamp to prevent EF Core from throwing a translation error when querying non-nullable `DateTime` fields.

In [ ]:
bool columnExists = false;
var connection = context.Database.GetDbConnection();
var connectionWasOpen = connection.State == System.Data.ConnectionState.Open;

if (!connectionWasOpen)
{
    await context.Database.OpenConnectionAsync();
}

try
{
    using var command = connection.CreateCommand();
    command.CommandText = "PRAGMA table_info(MediaLocations);";
    using var reader = await command.ExecuteReaderAsync();
    while (await reader.ReadAsync())
    {
        var nameCol = reader["name"]?.ToString();
        if (string.Equals(nameCol, "createdAt", StringComparison.OrdinalIgnoreCase))
        {
            columnExists = true;
            break;
        }
    }
}
finally
{
    if (!connectionWasOpen)
    {
        await context.Database.CloseConnectionAsync();
    }
}

if (!columnExists)
{
    Console.WriteLine("Column 'createdAt' not found. Adding column to MediaLocations table...");
    await context.Database.ExecuteSqlRawAsync("ALTER TABLE MediaLocations ADD COLUMN createdAt TEXT;");
    Console.WriteLine("✓ Added 'createdAt' column successfully.");
}
else
{
    Console.WriteLine("✓ Column 'createdAt' already exists in MediaLocations table.");
}

// Prevent EF Core null-reading exceptions on non-nullable CreatedAt properties
int rowsUpdated = await context.Database.ExecuteSqlRawAsync("UPDATE MediaLocations SET createdAt = datetime('now') WHERE createdAt IS NULL;");
if (rowsUpdated > 0)
{
    Console.WriteLine($"✓ Initialized {rowsUpdated} row(s) with NULL 'createdAt' to default timestamp.");
}

### Read and Update Media Locations with Disk Creation Time

In [ ]:
var locations = await context.MediaLocations.ToListAsync();
Console.WriteLine($"Found {locations.Count} locations in database.");

int updatedCount = 0;
foreach (var location in locations)
{
    DateTime creationTime;
    if (File.Exists(location.Path))
    {
        creationTime = File.GetCreationTimeUtc(location.Path);
        Console.WriteLine($"File exists: {location.Path} -> Created: {creationTime}");
    }
    else
    {
        creationTime = DateTime.UtcNow;
        Console.WriteLine($"File NOT found: {location.Path} -> Defaulting to: {creationTime}");
    }

    location.CreatedAt = creationTime;
    updatedCount++;
}

if (updatedCount > 0)
{
    await context.SaveChangesAsync();
    Console.WriteLine($"✓ Successfully updated {updatedCount} location(s).");
}
else
{
    Console.WriteLine("No locations found to update.");
}